# Vehicle Sensor Analytics & Predictive Maintenance
## Notebook 1: Exploratory Data Analysis

**Dataset:** Vehicle Maintenance Telemetry Data (Kaggle)  
**Signals:** Engine, Brake, Battery, Wheel Speed, Environmental  
**Targets:** `engine_failure_imminent`, `brake_issue_imminent`, `battery_issue_imminent`

---
### Sections
1. Setup & Load
2. Data Quality Check
3. Target Distribution
4. Engine Subsystem EDA
5. Brake Subsystem EDA
6. Battery Subsystem EDA
7. Correlation Analysis
8. Time-Series Trends
9. Brand-wise Analysis
10. Key Findings Summary

---
## 1. Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style='whitegrid', palette='muted')

import os
os.makedirs('../outputs', exist_ok=True)

In [ ]:
df = pd.read_csv('../data/vehicle_maintenance_telemetry_data.csv')

# Parse timestamps, drop placeholder failure_date
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.drop(columns=['failure_date'])  # 2050 placeholder, no signal
df = df.sort_values(['vehicle_id', 'timestamp']).reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df.timestamp.min()} → {df.timestamp.max()}')
print(f'Vehicles: {df.vehicle_id.nunique()}')
print(f'Brands: {df.brand.unique()}')

---
## 2. Data Quality Check

In [ ]:
# Null check
null_summary = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().mean() * 100).round(2)
})
null_summary = null_summary[null_summary.null_count > 0]

if null_summary.empty:
    print('No nulls found — clean dataset.')
else:
    print(null_summary)

In [ ]:
# Descriptive stats for numeric columns
numeric_cols = df.select_dtypes(include='number').columns.tolist()
targets = ['engine_failure_imminent', 'brake_issue_imminent', 'battery_issue_imminent']

df[numeric_cols].describe().T.round(2)

---
## 3. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
target_labels = ['Engine Failure\nImminent', 'Brake Issue\nImminent', 'Battery Issue\nImminent']
colors = ['#e74c3c', '#e67e22', '#2980b9']

for ax, col, label, color in zip(axes, targets, target_labels, colors):
    counts = df[col].value_counts()
    bars = ax.bar(['No Issue', 'Issue'], counts.values, color=['#bdc3c7', color], edgecolor='white', linewidth=1.2)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_ylabel('Count')
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{count}\n({count/len(df)*100:.1f}%)', ha='center', fontsize=10)

plt.suptitle('Class Distribution — Failure Targets', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/01_target_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# failure_type breakdown
print('Failure Type Counts:')
print(df['failure_type'].value_counts())
print(f'\nRows with any failure: {(df[targets].any(axis=1).sum())} ({df[targets].any(axis=1).mean()*100:.1f}%)')

---
## 4. Engine Subsystem EDA

In [ ]:
engine_cols = ['engine_temp_c', 'engine_rpm', 'engine_load_percent',
               'oil_pressure_psi', 'coolant_temp_c', 'exhaust_gas_temp_c',
               'fuel_consumption_lph', 'throttle_pos_percent']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, col in zip(axes, engine_cols):
    normal = df[df['engine_failure_imminent'] == 0][col]
    failure = df[df['engine_failure_imminent'] == 1][col]
    ax.hist(normal, bins=30, alpha=0.6, color='#2ecc71', label='Normal', density=True)
    ax.hist(failure, bins=30, alpha=0.6, color='#e74c3c', label='Failure', density=True)
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Engine Signals: Normal vs Engine Failure Imminent', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/02_engine_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots - median shift is easy to spot for interviews
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, col in zip(axes, engine_cols):
    df.boxplot(column=col, by='engine_failure_imminent', ax=ax,
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.set_xlabel('engine_failure_imminent (0=No, 1=Yes)')
    ax.set_ylabel('Value')

plt.suptitle('Engine Signal Boxplots by Failure Label', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/03_engine_boxplots.png', bbox_inches='tight')
plt.show()

---
## 5. Brake Subsystem EDA

In [ ]:
brake_cols = ['brake_fluid_level_psi', 'brake_pad_wear_mm', 'brake_temp_c',
              'abs_fault_indicator', 'brake_pedal_pos_percent']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, col in zip(axes, brake_cols):
    normal = df[df['brake_issue_imminent'] == 0][col]
    issue = df[df['brake_issue_imminent'] == 1][col]
    if col == 'abs_fault_indicator':
        # Categorical — use bar chart
        ct = df.groupby(['abs_fault_indicator', 'brake_issue_imminent']).size().unstack(fill_value=0)
        ct.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='white')
        ax.set_xticklabels(['No ABS Fault', 'ABS Fault'], rotation=0)
    else:
        ax.hist(normal, bins=30, alpha=0.6, color='#2ecc71', label='Normal', density=True)
        ax.hist(issue, bins=30, alpha=0.6, color='#e67e22', label='Issue', density=True)
        ax.legend(fontsize=8)
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.set_ylabel('Density')

plt.suptitle('Brake Signals: Normal vs Brake Issue Imminent', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/04_brake_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Wheel speed deviation — key feature for brake/ABS analysis
wheel_cols = ['wheel_speed_fl_kph', 'wheel_speed_fr_kph', 'wheel_speed_rl_kph', 'wheel_speed_rr_kph']
df['wheel_speed_mean'] = df[wheel_cols].mean(axis=1)
df['wheel_speed_std'] = df[wheel_cols].std(axis=1)  # High std = wheel mismatch = brake concern

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label in zip(axes,
                           ['wheel_speed_mean', 'wheel_speed_std'],
                           ['Mean Wheel Speed (kph)', 'Wheel Speed Std Dev (kph)']):
    for val, color, lbl in [(0, '#2ecc71', 'No Issue'), (1, '#e67e22', 'Brake Issue')]:
        ax.hist(df[df['brake_issue_imminent'] == val][col], bins=30,
                alpha=0.6, color=color, label=lbl, density=True)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Value')
    ax.legend()

plt.suptitle('Wheel Speed Features vs Brake Issue', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/05_wheel_speed_analysis.png', bbox_inches='tight')
plt.show()

---
## 6. Battery Subsystem EDA

In [ ]:
battery_cols = ['battery_voltage_v', 'battery_current_a', 'battery_temp_c',
                'alternator_output_v', 'battery_charge_percent', 'battery_health_percent']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, col in zip(axes, battery_cols):
    normal = df[df['battery_issue_imminent'] == 0][col]
    issue = df[df['battery_issue_imminent'] == 1][col]
    ax.hist(normal, bins=30, alpha=0.6, color='#2ecc71', label='Normal', density=True)
    ax.hist(issue, bins=30, alpha=0.6, color='#2980b9', label='Issue', density=True)
    ax.axvline(normal.mean(), color='green', linestyle='--', linewidth=1, label=f'Normal μ={normal.mean():.1f}')
    ax.axvline(issue.mean(), color='blue', linestyle='--', linewidth=1, label=f'Issue μ={issue.mean():.1f}')
    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.legend(fontsize=7)

plt.suptitle('Battery Signals: Normal vs Battery Issue Imminent', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/06_battery_distributions.png', bbox_inches='tight')
plt.show()

---
## 7. Correlation Analysis

In [ ]:
# Correlation of all numeric features with each target
feature_cols = [c for c in numeric_cols if c not in targets]

corr_with_targets = df[feature_cols + targets].corr()[targets].drop(targets)

fig, ax = plt.subplots(figsize=(8, 14))
sns.heatmap(corr_with_targets, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax,
            annot_kws={'size': 9})
ax.set_title('Feature Correlation with Failure Targets', fontsize=13, fontweight='bold')
ax.set_xticklabels(['Engine\nFailure', 'Brake\nIssue', 'Battery\nIssue'], fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/07_correlation_with_targets.png', bbox_inches='tight')
plt.show()

In [ ]:
# Inter-feature correlation heatmap (engine signals only to keep readable)
engine_battery_cols = engine_cols + battery_cols

fig, ax = plt.subplots(figsize=(14, 11))
corr_matrix = df[engine_battery_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # upper triangle mask
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Engine & Battery Signal Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/08_feature_correlation_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
# Top 5 correlated features per target
for target in targets:
    print(f'\nTop 5 features correlated with [{target}]:')
    top = corr_with_targets[target].abs().sort_values(ascending=False).head(5)
    for feat, val in top.items():
        print(f'  {feat:35s}: {corr_with_targets[target][feat]:+.3f}')

---
## 8. Time-Series Trends

In [ ]:
# Pick one vehicle for time-series visualization
sample_vehicle = df['vehicle_id'].value_counts().index[0]
vdf = df[df['vehicle_id'] == sample_vehicle].set_index('timestamp').sort_index()
print(f'Vehicle: {sample_vehicle} | Records: {len(vdf)} | Brand: {vdf.brand.iloc[0]}')

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)

# Engine temp + failure overlay
axes[0].plot(vdf.index, vdf['engine_temp_c'], color='#e74c3c', linewidth=1, label='Engine Temp (°C)')
axes[0].plot(vdf.index, vdf['coolant_temp_c'], color='#3498db', linewidth=1, alpha=0.7, label='Coolant Temp (°C)')
failure_mask = vdf['engine_failure_imminent'] == 1
axes[0].fill_between(vdf.index, vdf['engine_temp_c'].min(), vdf['engine_temp_c'].max(),
                     where=failure_mask, alpha=0.2, color='red', label='Failure Period')
axes[0].set_ylabel('Temperature (°C)')
axes[0].legend(fontsize=8)
axes[0].set_title(f'Vehicle {sample_vehicle} — Sensor Time Series', fontsize=12, fontweight='bold')

# RPM + throttle
ax2b = axes[1].twinx()
axes[1].plot(vdf.index, vdf['engine_rpm'], color='#2c3e50', linewidth=1, label='RPM')
ax2b.plot(vdf.index, vdf['throttle_pos_percent'], color='#f39c12', linewidth=1, alpha=0.7, label='Throttle %')
axes[1].set_ylabel('Engine RPM')
ax2b.set_ylabel('Throttle %')
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, fontsize=8)

# Battery voltage + charge
ax3b = axes[2].twinx()
axes[2].plot(vdf.index, vdf['battery_voltage_v'], color='#8e44ad', linewidth=1, label='Voltage (V)')
ax3b.plot(vdf.index, vdf['battery_charge_percent'], color='#27ae60', linewidth=1, alpha=0.7, label='Charge %')
battery_mask = vdf['battery_issue_imminent'] == 1
axes[2].fill_between(vdf.index, vdf['battery_voltage_v'].min(), vdf['battery_voltage_v'].max(),
                     where=battery_mask, alpha=0.2, color='purple', label='Battery Issue')
axes[2].set_ylabel('Battery Voltage (V)')
ax3b.set_ylabel('Charge %')
lines3, labels3 = axes[2].get_legend_handles_labels()
lines4, labels4 = ax3b.get_legend_handles_labels()
axes[2].legend(lines3 + lines4, labels3 + labels4, fontsize=8)

# Brake pad wear + brake temp
ax4b = axes[3].twinx()
axes[3].plot(vdf.index, vdf['brake_pad_wear_mm'], color='#e67e22', linewidth=1.5, label='Brake Pad Wear (mm)')
ax4b.plot(vdf.index, vdf['brake_temp_c'], color='#c0392b', linewidth=1, alpha=0.6, label='Brake Temp (°C)')
brake_mask = vdf['brake_issue_imminent'] == 1
axes[3].fill_between(vdf.index, vdf['brake_pad_wear_mm'].min(), vdf['brake_pad_wear_mm'].max(),
                     where=brake_mask, alpha=0.2, color='orange', label='Brake Issue')
axes[3].set_ylabel('Brake Pad Wear (mm)')
ax4b.set_ylabel('Brake Temp (°C)')
lines5, labels5 = axes[3].get_legend_handles_labels()
lines6, labels6 = ax4b.get_legend_handles_labels()
axes[3].legend(lines5 + lines6, labels5 + labels6, fontsize=8)
axes[3].set_xlabel('Timestamp')

plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../outputs/09_timeseries_single_vehicle.png', bbox_inches='tight')
plt.show()

---
## 9. Brand-wise Analysis

In [ ]:
# Failure rate per brand
brand_failure = df.groupby('brand')[targets].mean().mul(100).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
brand_failure.plot(kind='bar', ax=ax, color=['#e74c3c', '#e67e22', '#2980b9'],
                   edgecolor='white', linewidth=1)
ax.set_title('Failure Rate (%) by Vehicle Brand', fontsize=13, fontweight='bold')
ax.set_xlabel('Brand')
ax.set_ylabel('Failure Rate (%)')
ax.set_xticklabels(brand_failure.index, rotation=30, ha='right')
ax.legend(['Engine Failure', 'Brake Issue', 'Battery Issue'], fontsize=9)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', fontsize=8, padding=2)
plt.tight_layout()
plt.savefig('../outputs/10_brand_failure_rates.png', bbox_inches='tight')
plt.show()

In [ ]:
# Average engine temp and battery health by brand
brand_stats = df.groupby('brand')[['engine_temp_c', 'battery_health_percent',
                                    'brake_pad_wear_mm', 'odometer_reading']].mean().round(2)
print('Average Signal Values by Brand:')
brand_stats

---
## 10. Key Findings Summary

In [ ]:
print('=' * 60)
print('EDA SUMMARY')
print('=' * 60)

print(f'\nDataset: {df.shape[0]} records | {df.vehicle_id.nunique()} vehicles | {df.brand.nunique()} brands')
print(f'Date range: {df.timestamp.min().date()} to {df.timestamp.max().date()}')

print('\nTarget Class Balance:')
for t in targets:
    pos = df[t].sum()
    print(f'  {t:35s}: {pos} positive ({pos/len(df)*100:.1f}%)')

print('\nTop correlated feature per target:')
for t in targets:
    top_feat = corr_with_targets[t].abs().idxmax()
    top_val = corr_with_targets[t][top_feat]
    print(f'  {t:35s}: {top_feat} (r={top_val:+.3f})')

print('\nEngineered features added:')
print('  wheel_speed_mean — average of 4 wheel speeds')
print('  wheel_speed_std  — deviation across wheels (brake health proxy)')

print('\nNext: Notebook 02 — Anomaly Detection & Rolling Features')
print('=' * 60)

In [ ]:
# Save enriched dataframe for next notebook
df.to_csv('../data/df_eda.csv', index=False)
print('Saved: ../data/df_eda.csv')